# CPDMI/TissueNet models versus the public InstanSeg model

This comparison notebook evaluates the CPDMI-only 0.5-µm and 0.325-µm checkpoints, the 0.325-µm CPDMI checkpoint trained with 384-pixel tiles, a mixed CPDMI+TissueNet 0.325-µm checkpoint, and the public `fluorescence_nuclei_and_cells` v0.1.1 model. Each model is evaluated at its configured physical scale with deterministic preprocessing, its configured inference tile size, explicit postprocessing settings, and the same object-matching metric. The seed peak distance is now specified in physical units and converted to pixels separately for each model scale.

It reports the CPDMI `Validation` records and the saved dataset's `Test` records separately. In the current combined dataset, the 30 CPDMI validation records are present, but the `Test` split contains TissueNet records rather than a CPDMI held-out test set. The notebook prints that provenance explicitly; do not interpret the Test result as CPDMI test performance.

The full evaluation is gated because the Test split currently contains 1,320 records and each added model requires another pass. Set `RUN_EVALUATION=True` on a CUDA allocation, or set `TEST_RECORD_LIMIT` for a smaller smoke run.

The paired comparisons are: CPDMI-050 versus public at 0.5 µm; CPDMI-0325/256 versus CPDMI-0325/384 to assess tile context; CPDMI-0325/256 versus mixed CPDMI+TissueNet at 0.325 µm; and 0.5 versus 0.325 for the CPDMI-only models. The coverage table below makes the available nuclear supervision explicit; the mixed model uses source-balanced sampling during training, so the larger TissueNet pool is not sampled in simple proportion to its record count.

In [ ]:
import ast
from contextlib import contextmanager
import gc
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

TRAINING_ROOT = Path(os.environ.get(
    'INSTANSEG_TRAINING_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg',
)).expanduser().resolve()
DATASET_DIR = TRAINING_ROOT / 'datasets'
DATASET_FILE = DATASET_DIR / 'segmentation_dataset.pth'
# InstanSeg stores some records (notably the saved Test split) as paths relative
# to this directory.  Set this before importing/calling get_image().
os.environ['INSTANSEG_DATASET_PATH'] = str(DATASET_DIR)
MODEL_ROOT = TRAINING_ROOT / 'models'
SOURCE_ROOT = Path(os.environ.get(
    'INSTANSEG_EVAL_SOURCE_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg/slurm_runs/'
    'instanseg_multihead_0325_20260825/source/instanseg',
)).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get(
    'INSTANSEG_SPLIT_COMPARISON_ROOT',
    str(TRAINING_ROOT / 'model_comparisons' / 'cpdmi_public_tissuenet_0325_t384_validation_test_scaled_seed_distance'),
)).expanduser().resolve()

MODEL_NAME = 'cpdmi_050_t256_w128_heavy_multihead'
CPDMI_0325_MODEL_NAME = 'cpdmi_0325_t256_w128_heavy_multihead'
CPDMI_0325_T384_MODEL_NAME = 'cpdmi_0325_t384_w128_heavy_multihead'
MIXED_MODEL_NAME = 'cpdmi_tissuenet_0325_t256_w128_heavy_drop_multihead'
CHECKPOINT_FILENAME = 'model_weights.pth'
PUBLIC_MODEL_NAME = 'fluorescence_nuclei_and_cells'
PUBLIC_MODEL_VERSION = '0.1.1'
PUBLIC_MODEL_LABEL = f'public_{PUBLIC_MODEL_NAME}_v{PUBLIC_MODEL_VERSION}'
MODEL_SPECS = [
    {'name': MODEL_NAME, 'label': MODEL_NAME, 'kind': 'trained', 'training_source': 'CPDMI_2023', 'requested_pixel_size_um': 0.5, 'inference_tile_size_px': 256},
    {'name': CPDMI_0325_MODEL_NAME, 'label': CPDMI_0325_MODEL_NAME, 'kind': 'trained', 'training_source': 'CPDMI_2023', 'requested_pixel_size_um': 0.325, 'inference_tile_size_px': 256},
    {'name': CPDMI_0325_T384_MODEL_NAME, 'label': CPDMI_0325_T384_MODEL_NAME, 'kind': 'trained', 'training_source': 'CPDMI_2023', 'requested_pixel_size_um': 0.325, 'inference_tile_size_px': 384},
    {'name': MIXED_MODEL_NAME, 'label': 'mixed_' + MIXED_MODEL_NAME, 'kind': 'trained', 'training_source': 'CPDMI_2023 + TissueNet', 'requested_pixel_size_um': 0.325, 'inference_tile_size_px': 256},
    {'name': PUBLIC_MODEL_LABEL, 'label': PUBLIC_MODEL_LABEL, 'kind': 'public', 'training_source': 'public release', 'requested_pixel_size_um': 0.5, 'inference_tile_size_px': 256},
]
MODEL_LABELS = [spec['label'] for spec in MODEL_SPECS]
PUBLIC_MODEL_CACHE_ROOT = Path(os.environ.get(
    'INSTANSEG_PUBLIC_MODEL_CACHE',
    str(TRAINING_ROOT / 'public_model_cache'),
)).expanduser().resolve()

VALIDATION_SPLIT = 'Validation'
VALIDATION_PARENT_DATASET = 'CPDMI_2023'
TEST_SPLIT = 'Test'
# None means all records in the saved Test split. Set 'CPDMI_2023' if a
# future dataset file contains a true CPDMI Test split.
TEST_PARENT_DATASET = None
EXPECTED_CPDMI_VALIDATION_COUNT = 30
TEST_RECORD_LIMIT = 500

RUN_EVALUATION = True
SAVE_ARTIFACTS = True
DEVICE_OVERRIDE = os.environ.get('INSTANSEG_EVAL_DEVICE')
PUBLIC_DEVICE_OVERRIDE = os.environ.get('INSTANSEG_PUBLIC_INFERENCE_DEVICE')
INFERENCE_TILE_SIZE = 256
INFERENCE_BATCH_SIZE = 1
INFERENCE_DIMENSION_MULTIPLE = 16
GALLERY_RECORDS_PER_SPLIT = 1

# Comparison settings are shared in physical terms where appropriate.
# `peak_distance` is represented below at the 0.5-µm reference scale and
# converted per model before inference.
POSTPROCESSING = {
    'mask_threshold': 0.53,
    'peak_distance': 4,
    'seed_threshold': 0.5,
    'overlap_threshold': 0.5,
    'mean_threshold': -10000.0,
    'window_size': 128,
    'min_size': 10,
    'cleanup_fragments': False,
    'max_seeds': 2000,
    'resolve_cell_and_nucleus': True,
}
REFERENCE_PIXEL_SIZE_UM = 0.5
SEED_PEAK_DISTANCE_UM = POSTPROCESSING['peak_distance'] * REFERENCE_PIXEL_SIZE_UM

def peak_distance_pixels(pixel_size_um):
    return max(1, int(round(SEED_PEAK_DISTANCE_UM / float(pixel_size_um))))

def postprocessing_for_pixel_size(pixel_size_um):
    settings = dict(POSTPROCESSING)
    settings['peak_distance'] = peak_distance_pixels(pixel_size_um)
    return settings

MODEL_POSTPROCESSING = {
    spec['label']: postprocessing_for_pixel_size(spec['requested_pixel_size_um'])
    for spec in MODEL_SPECS
}
THRESHOLDS = [round(float(x), 2) for x in np.linspace(0.5, 1.0, 10)]

print('Dataset:', DATASET_FILE)
print('Models:', ', '.join(MODEL_LABELS))
print('Results:', RESULTS_ROOT)
print('Evaluation enabled:', RUN_EVALUATION)
display(pd.DataFrame([
    {
        'model': spec['label'],
        'pixel_size_um': float(spec['requested_pixel_size_um']),
        'peak_distance_px': MODEL_POSTPROCESSING[spec['label']]['peak_distance'],
        'effective_peak_distance_um': MODEL_POSTPROCESSING[spec['label']]['peak_distance'] * float(spec['requested_pixel_size_um']),
    }
    for spec in MODEL_SPECS
]))

## Load the matching InstanSeg source and dataset

In [ ]:
if not (SOURCE_ROOT / 'instanseg').is_dir():
    raise FileNotFoundError(
        f'Expected InstanSeg source at {SOURCE_ROOT / "instanseg"}. '
        'Set INSTANSEG_EVAL_SOURCE_ROOT and restart the kernel.'
    )
if not DATASET_FILE.is_file():
    raise FileNotFoundError(DATASET_FILE)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import torch
import instanseg
from instanseg import InstanSeg as PublicInstanSeg
from instanseg.utils.augmentations import Augmentations
from instanseg.utils.data_loader import get_image
from instanseg.utils.loss import instanseg_loss as loss_utils
from instanseg.utils.loss.instanseg_loss import InstanSeg as LossInstanSeg
from instanseg.utils.metrics import matching_torch
from instanseg.utils.model_loader import (
    build_model_from_dict,
    has_adaptor_net_state_dict,
    has_pixel_classifier_model,
    has_pixel_classifier_state_dict,
    read_model_args_from_csv,
    remove_module_prefix_from_dict,
)
from instanseg.utils.models.ChannelInvariantNet import AdaptorNetWrapper, has_AdaptorNet
from instanseg.utils.tiling import _instanseg_padding, _recover_padding, _sliding_window_inference

instanseg_import_path = Path(instanseg.__file__).resolve()
if not instanseg_import_path.is_relative_to(SOURCE_ROOT / 'instanseg'):
    raise RuntimeError(f'Imported InstanSeg from {instanseg_import_path}, not {SOURCE_ROOT}.')

DEVICE = DEVICE_OVERRIDE or ('cuda:0' if torch.cuda.is_available() else 'cpu')
PUBLIC_DEVICE = PUBLIC_DEVICE_OVERRIDE or DEVICE
print('InstanSeg source:', instanseg_import_path)
print('PyTorch:', torch.__version__)
print('Trained device:', DEVICE)
print('Public device:', PUBLIC_DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(torch.cuda.current_device()))

## Select and document the two evaluation sets

In [ ]:
def _safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def _resolve_array(value):
    if isinstance(value, (str, Path)):
        return np.asarray(get_image(str(value)))
    return np.asarray(value)

def _nucleus_value(item):
    return item.get('nucleus_masks') if item.get('nucleus_masks') is not None else item.get('masks')

def _cell_value(item):
    return item.get('cell_masks')

def _record_id(split, index, item):
    return f'{split}[{index}]::{item.get("parent_dataset", "<missing>")}::{item.get("filename", index)}'

def _record_info(split, index, item):
    nucleus = _nucleus_value(item)
    cell = _cell_value(item)
    image = _resolve_array(item['image'])
    return {
        'split': split,
        'source_index': int(index),
        'record_id': _record_id(split, index, item),
        'filename': str(item.get('filename', index)),
        'parent_dataset': str(item.get('parent_dataset', '')),
        'platform': str(item.get('platform', '')),
        'native_pixel_size_um': float(item['pixel_size']),
        'image_shape': tuple(int(v) for v in image.shape),
        'nucleus_annotation': nucleus is not None,
        'cell_annotation': cell is not None,
    }

def select_records(dataset, split, parent_dataset=None, limit=None):
    if split not in dataset:
        raise KeyError(f'Missing split {split!r}; available splits: {sorted(dataset)}')
    selected = []
    for index, item in enumerate(dataset[split]):
        if parent_dataset is not None and item.get('parent_dataset') != parent_dataset:
            continue
        if item.get('duplicate', False):
            continue
        if _nucleus_value(item) is None and _cell_value(item) is None:
            continue
        selected.append((index, item))
    if limit is not None:
        selected = selected[:int(limit)]
    return [(_record_info(split, index, item), item) for index, item in selected]

dataset = _safe_torch_load(DATASET_FILE)
validation_records = select_records(
    dataset, VALIDATION_SPLIT, VALIDATION_PARENT_DATASET,
)
test_records = select_records(
    dataset, TEST_SPLIT, TEST_PARENT_DATASET, TEST_RECORD_LIMIT,
)
if len(validation_records) != EXPECTED_CPDMI_VALIDATION_COUNT:
    raise RuntimeError(
        f'Expected {EXPECTED_CPDMI_VALIDATION_COUNT} CPDMI validation records, '
        f'found {len(validation_records)}.'
    )

RECORD_GROUPS = {
    'CPDMI validation': validation_records,
    'Saved Test split': test_records,
}
RECORDS = [record for group in RECORD_GROUPS.values() for record in group]
selection_table = pd.DataFrame([info for info, _ in RECORDS])
display(selection_table.head(10))
print('Splits:', {name: len(records) for name, records in RECORD_GROUPS.items()})
print('Test parent datasets:', selection_table[selection_table['split'] == TEST_SPLIT]['parent_dataset'].value_counts().to_dict())
print('Test annotation coverage:', selection_table[selection_table['split'] == TEST_SPLIT][['nucleus_annotation', 'cell_annotation']].sum().to_dict())

coverage_rows = []
for split_name, split_items in dataset.items():
    parents = sorted({str(item.get('parent_dataset', '')) for item in split_items})
    for parent in parents:
        parent_items = [item for item in split_items if str(item.get('parent_dataset', '')) == parent and not item.get('duplicate', False)]
        coverage_rows.append({
            'split': split_name, 'parent_dataset': parent, 'records': len(parent_items),
            'nucleus_labeled_records': sum(_nucleus_value(item) is not None for item in parent_items),
            'cell_labeled_records': sum(_cell_value(item) is not None for item in parent_items),
        })
print('Annotation coverage in the saved combined dataset:')
display(pd.DataFrame(coverage_rows))

## Deterministic preparation at each model's physical scale

In [ ]:
AUGMENTER = Augmentations()

def _labels_from_item(item):
    nucleus = _nucleus_value(item)
    cell = _cell_value(item)
    reference = _resolve_array(nucleus if nucleus is not None else cell)
    if nucleus is None:
        nucleus_array = np.full(reference.shape, -1, dtype=np.int32)
    else:
        nucleus_array = _resolve_array(nucleus).astype(np.int32, copy=False)
    if cell is None:
        cell_array = np.full(reference.shape, -1, dtype=np.int32)
    else:
        cell_array = _resolve_array(cell).astype(np.int32, copy=False)
    if nucleus_array.shape != cell_array.shape:
        raise ValueError(f'Label shape mismatch for {item.get("filename", "<unnamed>")}')
    return np.stack((nucleus_array, cell_array), axis=0)

def prepare_record(item, requested_pixel_size_um):
    image = _resolve_array(item['image'])
    labels = _labels_from_item(item)
    image_tensor, labels_tensor = AUGMENTER.to_tensor(image, labels, normalize=False)
    image_tensor, _ = AUGMENTER.normalize(image_tensor)
    image_tensor, labels_tensor = AUGMENTER.torch_rescale(
        image_tensor, labels_tensor,
        current_pixel_size=float(item['pixel_size']),
        requested_pixel_size=float(requested_pixel_size_um),
        crop=False,
        modality='Fluorescence',
    )
    image_tensor = image_tensor.contiguous().float()
    labels_tensor = labels_tensor.contiguous().to(torch.int32)
    if image_tensor.shape[-2:] != labels_tensor.shape[-2:]:
        raise ValueError('Prepared image and label shapes differ.')
    return image_tensor, labels_tensor

example_info, example_item = RECORDS[0]
example_image, example_labels = prepare_record(example_item, 0.5)
print('Example:', example_info['record_id'])
print('Prepared image:', tuple(example_image.shape), example_image.dtype)
print('Prepared labels:', tuple(example_labels.shape), example_labels.dtype)

## Load the trained checkpoints and public model

In [ ]:
from torch import nn

def _optional_float(value):
    return None if isinstance(value, (float, np.floating)) and np.isnan(value) else value

def load_trained_model(model_name, device=None):
    device = device or DEVICE
    config = read_model_args_from_csv(path=MODEL_ROOT, folder=model_name)
    checkpoint_path = MODEL_ROOT / model_name / CHECKPOINT_FILENAME
    checkpoint = _safe_torch_load(checkpoint_path)
    state = remove_module_prefix_from_dict(checkpoint['model_state_dict'])
    method = LossInstanSeg(
        n_sigma=int(config['n_sigma']),
        binary_loss_fn_str=str(config['binary_loss_fn']),
        seed_loss_fn=str(config['seed_loss_fn']),
        device=device,
        cells_and_nuclei=bool(config['cells_and_nuclei']),
        window_size=int(config['window_size']),
        dim_coords=int(config['dim_coords']),
        dim_seeds=int(config['dim_seeds']),
        feature_engineering_function=str(config['feature_engineering']),
        bg_weight=_optional_float(config.get('bg_weight')),
    )
    model = build_model_from_dict(config, random_seed=None)
    if has_pixel_classifier_state_dict(state) and not has_pixel_classifier_model(model):
        model = method.initialize_pixel_classifier(model, MLP_width=int(config.get('mlp_width', 5)))
    if has_adaptor_net_state_dict(state) and not has_AdaptorNet(model):
        model = AdaptorNetWrapper(
            model, norm=config.get('norm'), adaptor_net_str=str(config.get('adaptor_net_str', '1'))
        )
    model.load_state_dict(state, strict=True)
    model = model.to(device).eval()
    del checkpoint, state
    return model, config, method

def load_public_model(device=None):
    device = device or PUBLIC_DEVICE
    model_index_path = SOURCE_ROOT / 'instanseg' / 'bioimageio_models' / 'model-index.json'
    entries = [entry for entry in json.loads(model_index_path.read_text()) if entry.get('name') == PUBLIC_MODEL_NAME]
    if not entries or entries[0].get('version') != PUBLIC_MODEL_VERSION:
        raise RuntimeError(f'Public model {PUBLIC_MODEL_NAME} v{PUBLIC_MODEL_VERSION} is unavailable.')
    PUBLIC_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['INSTANSEG_BIOIMAGEIO_PATH'] = str(PUBLIC_MODEL_CACHE_ROOT)
    runner = PublicInstanSeg(
        model_type=PUBLIC_MODEL_NAME, device=device, verbosity=0, channels_last=False
    )
    network = runner.instanseg.eval()
    if abs(float(network.pixel_size) - 0.5) > 1e-6:
        raise ValueError(f'Public model pixel size is {network.pixel_size}, not 0.5.')
    return runner, network

def load_model_bundle(spec):
    if spec['kind'] == 'trained':
        model, config, method = load_trained_model(spec['name'], device=DEVICE)
        expected = float(spec['requested_pixel_size_um'])
        actual = float(config['requested_pixel_size'])
        if abs(actual - expected) > 1e-6:
            raise ValueError(f'{spec["name"]} requests {actual} µm, expected {expected} µm.')
        return {
            'spec': spec, 'model': model, 'config': config, 'method': method,
            'predictor': RawTilePredictor(model, method.postprocessing),
            'device': DEVICE,
        }
    if spec['kind'] == 'public':
        runner, network = load_public_model(device=PUBLIC_DEVICE)
        return {
            'spec': spec, 'runner': runner, 'network': network,
            'predictor': PublicPythonPredictor(network),
            'device': PUBLIC_DEVICE,
        }
    raise ValueError(f'Unknown model kind: {spec["kind"]}')

print('Model loading functions ready.')

## Shared tiled prediction

The public network's sparse TorchScript postprocessor previously failed on CUDA symmetry checks and on CPU allocation. The wrapper below runs the public raw network and applies the equivalent Python postprocessor with coalesced, symmetrized IoU adjacency.

In [ ]:
def _pad_for_model(image_tensor):
    original_shape = tuple(image_tensor.shape[-2:])
    pad_height = (-original_shape[0]) % INFERENCE_DIMENSION_MULTIPLE
    pad_width = (-original_shape[1]) % INFERENCE_DIMENSION_MULTIPLE
    if pad_height == 0 and pad_width == 0:
        return image_tensor, original_shape
    mode = 'replicate' if pad_height >= original_shape[0] or pad_width >= original_shape[1] else 'reflect'
    return torch.nn.functional.pad(image_tensor, (0, pad_width, 0, pad_height), mode=mode), original_shape

def _crop_prediction(prediction, original_shape):
    return prediction[..., :original_shape[0], :original_shape[1]]

class RawTilePredictor(nn.Module):
    def __init__(self, model, postprocessor):
        super().__init__()
        self.model = model
        self.postprocessor = postprocessor

    def forward(self, image_batch, resolve_cell_and_nucleus=True, **kwargs):
        raw_output = self.model(image_batch)
        if isinstance(raw_output, (list, tuple)):
            raw_output = raw_output[0]
        labels = torch.stack([self.postprocessor(output, **kwargs) for output in raw_output])
        if resolve_cell_and_nucleus and labels.ndim == 4 and labels.shape[1] == 2:
            labels = loss_utils.resolve_cell_and_nucleus_boundaries(labels.float())
        return labels

def predict_trained(predictor, image_tensor, device=None, tile_size=None, postprocessing=None):
    device = device or DEVICE
    tile_size = int(tile_size or INFERENCE_TILE_SIZE)
    postprocessing = dict(postprocessing or POSTPROCESSING)
    model_input, original_shape = _pad_for_model(image_tensor)
    with torch.inference_mode():
        prediction = _sliding_window_inference(
            model_input, predictor,
            window_size=(tile_size, tile_size),
            sw_device=device, device='cpu', batch_size=INFERENCE_BATCH_SIZE,
            output_channels=2, show_progress=False, instanseg_kwargs=postprocessing,
        )
    return _crop_prediction(prediction.squeeze(0).to(torch.int32), original_shape)

ORIGINAL_FAST_SPARSE_IOU = loss_utils.fast_sparse_iou
ORIGINAL_FIND_CONNECTED_COMPONENTS = loss_utils.find_connected_components

def _safe_fast_sparse_iou(sparse_onehot):
    sparse_onehot = sparse_onehot.coalesce()
    intersection = torch.sparse.mm(sparse_onehot, sparse_onehot.T).to_dense()
    sizes = torch.sparse.sum(sparse_onehot, dim=(1,))[None].to_dense()
    union = sizes.T + sizes - intersection
    iou = intersection / union
    return (iou + iou.T) * 0.5

def _safe_find_connected_components(adjacency_matrix, max_iterations=100):
    symmetric = torch.logical_or(adjacency_matrix != 0, adjacency_matrix.T != 0).to(adjacency_matrix.dtype)
    return ORIGINAL_FIND_CONNECTED_COMPONENTS(symmetric, max_iterations)

@contextmanager
def _safe_public_postprocessing():
    previous_iou = loss_utils.fast_sparse_iou
    previous_components = loss_utils.find_connected_components
    loss_utils.fast_sparse_iou = _safe_fast_sparse_iou
    loss_utils.find_connected_components = _safe_find_connected_components
    try:
        yield
    finally:
        loss_utils.fast_sparse_iou = previous_iou
        loss_utils.find_connected_components = previous_components

class PublicPythonPredictor(nn.Module):
    def __init__(self, network):
        super().__init__()
        self.network = network
        public_device = str(next(iter(network.parameters())).device)
        self.postprocessor = LossInstanSeg(
            n_sigma=int(network.n_sigma), binary_loss_fn_str='lovasz_hinge', seed_loss_fn='binary_xloss',
            device=public_device, cells_and_nuclei=bool(network.cells_and_nuclei),
            window_size=int(network.default_window_size), dim_coords=int(network.dim_coords),
            dim_seeds=int(network.dim_seeds), feature_engineering_function='0',
        )
        self.postprocessor.feature_engineering = loss_utils.feature_engineering_slow
        self.postprocessor.pixel_classifier = network.pixel_classifier

    def forward(self, image_batch, resolve_cell_and_nucleus=True, **kwargs):
        with torch.amp.autocast(image_batch.device.type, enabled=False):
            model_input = image_batch.clamp(min=-2, max=3)
            model_input, padding = _instanseg_padding(model_input, extra_pad=0)
            raw_output = self.network.fcn(model_input)
            raw_output = _recover_padding(raw_output, padding)
            with _safe_public_postprocessing():
                labels = [
                    self.postprocessor.postprocessing(
                        raw_sample, device=image_batch.device, classifier=self.network.pixel_classifier, **kwargs
                    ) for raw_sample in raw_output
                ]
            labels = torch.stack(labels)
            if resolve_cell_and_nucleus and labels.shape[1] == 2:
                labels = loss_utils.resolve_cell_and_nucleus_boundaries(labels.float())
            return labels.float()

def predict_public(predictor, image_tensor, device=None, tile_size=None, postprocessing=None):
    device = device or PUBLIC_DEVICE
    tile_size = int(tile_size or INFERENCE_TILE_SIZE)
    postprocessing = dict(postprocessing or POSTPROCESSING)
    model_input, original_shape = _pad_for_model(image_tensor)
    with torch.inference_mode():
        prediction = _sliding_window_inference(
            model_input, predictor,
            window_size=(tile_size, tile_size),
            sw_device=device, device='cpu', batch_size=INFERENCE_BATCH_SIZE,
            output_channels=2, show_progress=False, instanseg_kwargs=postprocessing,
        )
    return _crop_prediction(prediction.squeeze(0).to(torch.int32), original_shape)

print('Prediction functions ready.')

## Run the comparison

In [ ]:
TARGET_NAMES = ('nuclei', 'cells')

def _evaluation_arrays(gt, prediction, target_index):
    gt_eval = gt[target_index].clone().to(torch.int32)
    prediction_eval = prediction[target_index].clone().to(torch.int32)
    valid = gt_eval >= 0
    if not bool(valid.all()):
        gt_eval = gt_eval.clone()
        prediction_eval = prediction_eval.clone()
        gt_eval[~valid] = 0
        prediction_eval[~valid] = 0
    if not bool((gt_eval > 0).any()):
        return None
    return gt_eval, prediction_eval

def metric_rows(split_label, model_name, info, gt, prediction):
    rows = []
    for target_index, target_name in enumerate(TARGET_NAMES):
        arrays = _evaluation_arrays(gt, prediction, target_index)
        if arrays is None:
            continue
        gt_eval, prediction_eval = arrays
        for threshold, stat in zip(THRESHOLDS, matching_torch(gt_eval, prediction_eval, THRESHOLDS)):
            rows.append({
                'evaluation_set': split_label, 'split': info['split'], 'model': model_name,
                'record_id': info['record_id'], 'filename': info['filename'],
                'parent_dataset': info['parent_dataset'], 'target': target_name,
                'threshold': float(threshold), 'tp': int(stat.tp), 'fp': int(stat.fp), 'fn': int(stat.fn),
                'precision': float(stat.precision), 'recall': float(stat.recall), 'f1': float(stat.f1),
                'n_true': int(stat.n_true), 'n_pred': int(stat.n_pred),
            })
    return rows

def aggregate_metrics(per_record):
    if per_record.empty:
        return pd.DataFrame(), pd.DataFrame()
    threshold_rows = []
    for keys, group in per_record.groupby(['evaluation_set', 'model', 'target', 'threshold'], sort=True):
        evaluation_set, model, target, threshold = keys
        tp, fp, fn = (int(group[name].sum()) for name in ('tp', 'fp', 'fn'))
        threshold_rows.append({
            'evaluation_set': evaluation_set, 'model': model, 'target': target, 'threshold': threshold,
            'tp': tp, 'fp': fp, 'fn': fn, 'precision': tp / max(tp + fp, 1e-10),
            'recall': tp / max(tp + fn, 1e-10), 'f1': 2 * tp / max(2 * tp + fp + fn, 1e-10),
            'macro_f1': float(group['f1'].mean()), 'n_true': int(group['n_true'].sum()),
            'n_pred': int(group['n_pred'].sum()), 'n_records': int(group['record_id'].nunique()),
        })
    threshold_table = pd.DataFrame(threshold_rows)
    at_05 = threshold_table[threshold_table['threshold'] == 0.5].copy()
    summary = at_05.rename(columns={
        'precision': 'precision_iou50', 'recall': 'recall_iou50', 'f1': 'f1_iou50',
        'macro_f1': 'macro_f1_iou50',
    })
    return summary, threshold_table

def _display_base(image_tensor):
    base = image_tensor.numpy().mean(axis=0)
    low, high = np.percentile(base, (1, 99))
    return np.clip((base - low) / max(float(high - low), 1e-6), 0, 1)

def _overlay(base, labels):
    from skimage.segmentation import find_boundaries
    labels = np.asarray(labels).copy()
    labels[labels < 0] = 0
    rgb = np.repeat(base[..., None], 3, axis=-1)
    rgb[find_boundaries(labels[1], mode='outer')] = (1.0, 0.2, 0.1)
    rgb[find_boundaries(labels[0], mode='outer')] = (0.1, 0.8, 1.0)
    return rgb

SUMMARY_DF = pd.DataFrame()
THRESHOLD_DF = pd.DataFrame()
PER_RECORD_DF = pd.DataFrame()
GALLERY = {}

if not RUN_EVALUATION:
    print('Evaluation is gated. Set RUN_EVALUATION=True and rerun this cell on CUDA.')
else:
    all_rows = []
    gallery_record_ids = {
        set_name: {info['record_id'] for info, _ in records[:GALLERY_RECORDS_PER_SPLIT]}
        for set_name, records in RECORD_GROUPS.items()
    }
    GALLERY = {}
    started = time.perf_counter()
    for spec in MODEL_SPECS:
        label = spec['label']
        print(f'[{label}] target scale={spec["requested_pixel_size_um"]} µm, tile={spec["inference_tile_size_px"]} px')
        bundle = load_model_bundle(spec)
        try:
            for set_name, records in RECORD_GROUPS.items():
                print(f'  [{set_name}] {len(records)} records')
                for number, (info, item) in enumerate(records, start=1):
                    image_tensor, ground_truth = prepare_record(item, spec['requested_pixel_size_um'])
                    postprocessing = MODEL_POSTPROCESSING[label]
                    if spec['kind'] == 'trained':
                        prediction = predict_trained(bundle['predictor'], image_tensor, bundle['device'], spec['inference_tile_size_px'], postprocessing=postprocessing).cpu()
                    else:
                        prediction = predict_public(bundle['predictor'], image_tensor, bundle['device'], spec['inference_tile_size_px'], postprocessing=postprocessing).cpu()
                    all_rows.extend(metric_rows(set_name, label, info, ground_truth, prediction))
                    if info['record_id'] in gallery_record_ids[set_name]:
                        gallery_key = (set_name, info['record_id'])
                        GALLERY.setdefault(gallery_key, {'samples': {}})['samples'][label] = {
                            'image': image_tensor.cpu(), 'ground_truth': ground_truth.cpu(),
                            'prediction': prediction, 'target_pixel_size_um': spec['requested_pixel_size_um'],
                        }
                    if number == 1 or number % 25 == 0 or number == len(records):
                        print(f'    {number}/{len(records)}')
        finally:
            del bundle
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    PER_RECORD_DF = pd.DataFrame(all_rows)
    SUMMARY_DF, THRESHOLD_DF = aggregate_metrics(PER_RECORD_DF)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes.')
    display(SUMMARY_DF[['evaluation_set', 'model', 'target', 'n_records', 'n_true', 'n_pred', 'precision_iou50', 'recall_iou50', 'f1_iou50', 'macro_f1_iou50']])
    if SAVE_ARTIFACTS:
        RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
        PER_RECORD_DF.to_csv(RESULTS_ROOT / 'per_record.csv', index=False)
        SUMMARY_DF.to_csv(RESULTS_ROOT / 'summary.csv', index=False)
        THRESHOLD_DF.to_csv(RESULTS_ROOT / 'thresholds.csv', index=False)
        manifest = {
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'dataset_file': str(DATASET_FILE), 'source_root': str(SOURCE_ROOT),
            'models': MODEL_SPECS,
            'checkpoint_filename': CHECKPOINT_FILENAME,
            'target_pixel_sizes_um': sorted({float(spec['requested_pixel_size_um']) for spec in MODEL_SPECS}),
            'inference_tile_size_px': {spec['label']: spec['inference_tile_size_px'] for spec in MODEL_SPECS},
            'postprocessing_reference': POSTPROCESSING,
            'postprocessing_by_model': MODEL_POSTPROCESSING,
            'records': [info for info, _ in RECORDS],
        }
        (RESULTS_ROOT / 'provenance.json').write_text(json.dumps(manifest, indent=2, default=str) + '\n')
        print('Saved:', RESULTS_ROOT)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Results and qualitative spot checks

Metrics are pooled by object count and also reported as macro F1. Models at 0.325 µm are compared on the same underlying records, but their raster grids differ from the 0.5-µm models.

In [ ]:
if SUMMARY_DF.empty:
    summary_path = RESULTS_ROOT / 'summary.csv'
    if summary_path.is_file():
        SUMMARY_DF = pd.read_csv(summary_path)
        print('Loaded saved summary:', summary_path)
    else:
        print('No results in memory or on disk yet.')
if not SUMMARY_DF.empty:
    display(SUMMARY_DF[['evaluation_set', 'model', 'target', 'n_records', 'n_true', 'n_pred', 'f1_iou50', 'macro_f1_iou50']])

if not THRESHOLD_DF.empty:
    figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for axis, target in zip(axes, TARGET_NAMES):
        data = THRESHOLD_DF[THRESHOLD_DF['target'] == target]
        for (evaluation_set, model), curve in data.groupby(['evaluation_set', 'model']):
            axis.plot(curve['threshold'], curve['f1'], marker='o', label=f'{evaluation_set}: {model}')
        axis.set_title(target.title())
        axis.set_xlabel('IoU threshold')
        axis.set_ylabel('pooled/object-count F1')
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.25)
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    figure.tight_layout()
    plt.show()

for (set_name, record_id), gallery_entry in GALLERY.items():
    samples = gallery_entry['samples']
    first_sample = next(iter(samples.values()))
    figure, axes = plt.subplots(1, len(samples) + 1, figsize=(4 * (len(samples) + 1), 4), squeeze=False)
    axes = axes[0]
    first_base = _display_base(first_sample['image'])
    axes[0].imshow(_overlay(first_base, first_sample['ground_truth']))
    axes[0].set_title(f'{set_name}\nground truth @ {first_sample["target_pixel_size_um"]} µm')
    for axis_index, (label, sample) in enumerate(samples.items(), start=1):
        base = _display_base(sample['image'])
        axes[axis_index].imshow(_overlay(base, sample['prediction']))
        axes[axis_index].set_title(f'{label}\n{sample["target_pixel_size_um"]} µm')
    for axis in axes: axis.axis('off')
    figure.suptitle(record_id, y=1.02)
    figure.tight_layout()
    plt.show()

## Interpretation

- Read the CPDMI validation rows as the direct 0.5-µm CPDMI/public comparison and the 0.325-µm CPDMI/TissueNet comparisons.
- The saved Test split is a cross-dataset TissueNet evaluation in the current dataset file, not a CPDMI test result.
- `f1_iou50` pools object matches across records; `macro_f1_iou50` gives every record equal weight.
- The 256-tile models share a 256-pixel inference tile; the `cpdmi_0325_t384_w128_heavy_multihead` row uses a 384-pixel inference tile to match its training configuration. The 0.325-µm models are compared on the same underlying records but on a finer raster grid than the 0.5-µm models.
- The public model is run through its raw network plus the safe Python postprocessor described above; that implementation detail is retained in `provenance.json`.

In [ ]:
# GPU evaluation of the independent CPDMI CODEX holdout.
# Run this cell after the setup/function cells above. It does not use the saved TissueNet Test split.
CODEX_RAW_ROOT = Path(os.environ.get(
    'INSTANSEG_CODEX_RAW_ROOT',
    str(TRAINING_ROOT / 'Raw_Datasets'),
)).expanduser().resolve()
CODEX_RESULTS_ROOT = Path(os.environ.get(
    'INSTANSEG_CODEX_RESULTS_ROOT',
    str(TRAINING_ROOT / 'model_comparisons' / 'cpdmi_codex_holdout_all_models'),
)).expanduser().resolve()
CODEX_DEVICE = os.environ.get('INSTANSEG_CODEX_DEVICE', 'cuda:0')
if not torch.cuda.is_available():
    raise RuntimeError(
        'This holdout cell is intentionally GPU-only. Select a CUDA notebook kernel '
        'or set up a GPU allocation before running it.'
    )
os.environ['INSTANSEG_RAW_DATASETS'] = str(CODEX_RAW_ROOT)
from instanseg.utils.data_download import load_CPDMI_CODEX
codex_records = load_CPDMI_CODEX({'Train': []})['Train']
if len(codex_records) != 10:
    raise RuntimeError(f'Expected 10 downloaded CODEX crops, found {len(codex_records)}.')
print('CODEX raw root:', CODEX_RAW_ROOT)
print('Records:', len(codex_records))
print('Nuclear records:', sum('nucleus_masks' in item for item in codex_records))
display(pd.DataFrame([{
    'record_id': item['filename'],
    'image_shape': tuple(item['image'].shape),
    'pixel_size_um': item['pixel_size'],
    'cell_objects': int(item['cell_masks'].max()),
    'nucleus_objects': int(item['nucleus_masks'].max()) if 'nucleus_masks' in item else None,
} for item in codex_records]))

# Make this holdout cell independent of the large saved-Test evaluation cell.
if 'metric_rows' not in globals():
    TARGET_NAMES = ('nuclei', 'cells')
    def _evaluation_arrays(gt, prediction, target_index):
        gt_eval = gt[target_index].clone().to(torch.int32)
        prediction_eval = prediction[target_index].clone().to(torch.int32)
        valid = gt_eval >= 0
        if not bool(valid.all()):
            gt_eval = gt_eval.clone(); prediction_eval = prediction_eval.clone()
            gt_eval[~valid] = 0; prediction_eval[~valid] = 0
        if not bool((gt_eval > 0).any()):
            return None
        return gt_eval, prediction_eval
    def metric_rows(split_label, model_name, info, gt, prediction):
        rows = []
        for target_index, target_name in enumerate(TARGET_NAMES):
            arrays = _evaluation_arrays(gt, prediction, target_index)
            if arrays is None:
                continue
            gt_eval, prediction_eval = arrays
            for threshold, stat in zip(THRESHOLDS, matching_torch(gt_eval, prediction_eval, THRESHOLDS)):
                rows.append({
                    'evaluation_set': split_label, 'split': info['split'], 'model': model_name,
                    'record_id': info['record_id'], 'filename': info['filename'],
                    'parent_dataset': info['parent_dataset'], 'target': target_name,
                    'threshold': float(threshold), 'tp': int(stat.tp), 'fp': int(stat.fp), 'fn': int(stat.fn),
                    'precision': float(stat.precision), 'recall': float(stat.recall), 'f1': float(stat.f1),
                    'n_true': int(stat.n_true), 'n_pred': int(stat.n_pred),
                })
        return rows
if 'aggregate_metrics' not in globals():
    def aggregate_metrics(per_record):
        threshold_rows = []
        for keys, group in per_record.groupby(['evaluation_set', 'model', 'target', 'threshold'], sort=True):
            evaluation_set, model, target, threshold = keys
            tp, fp, fn = (int(group[name].sum()) for name in ('tp', 'fp', 'fn'))
            threshold_rows.append({
                'evaluation_set': evaluation_set, 'model': model, 'target': target, 'threshold': threshold,
                'tp': tp, 'fp': fp, 'fn': fn, 'precision': tp / max(tp + fp, 1e-10),
                'recall': tp / max(tp + fn, 1e-10), 'f1': 2 * tp / max(2 * tp + fp + fn, 1e-10),
                'macro_f1': float(group['f1'].mean()), 'n_true': int(group['n_true'].sum()),
                'n_pred': int(group['n_pred'].sum()), 'n_records': int(group['record_id'].nunique()),
            })
        threshold_table = pd.DataFrame(threshold_rows)
        at_05 = threshold_table[threshold_table['threshold'] == 0.5].copy()
        summary = at_05.rename(columns={
            'precision': 'precision_iou50', 'recall': 'recall_iou50', 'f1': 'f1_iou50',
            'macro_f1': 'macro_f1_iou50',
        })
        return summary, threshold_table

# Reuse the controlled comparison path: each model's target scale and configured
# inference tile size, plus the same explicit postprocessing as the CPDMI comparison.
DEVICE = CODEX_DEVICE
PUBLIC_DEVICE = CODEX_DEVICE
codex_rows = []
codex_gallery = {}
started = time.perf_counter()
for spec in MODEL_SPECS:
    label = spec['label']
    print(f'[{label}] target scale={spec["requested_pixel_size_um"]} µm, tile={spec["inference_tile_size_px"]} px', flush=True)
    bundle = load_model_bundle(spec)
    try:
        for number, item in enumerate(codex_records, start=1):
            record_id = str(item['filename'])
            info = {
                'split': 'CODEX', 'source_index': number - 1, 'record_id': record_id,
                'filename': record_id, 'parent_dataset': 'CPDMI_2023', 'platform': 'CODEX',
                'native_pixel_size_um': float(item['pixel_size']),
            }
            image_tensor, ground_truth = prepare_record(item, spec['requested_pixel_size_um'])
            postprocessing = MODEL_POSTPROCESSING[spec['label']]
            if spec['kind'] == 'trained':
                prediction = predict_trained(bundle['predictor'], image_tensor, bundle['device'], spec['inference_tile_size_px'], postprocessing=postprocessing).cpu()
            else:
                prediction = predict_public(bundle['predictor'], image_tensor, bundle['device'], spec['inference_tile_size_px'], postprocessing=postprocessing).cpu()
            codex_rows.extend(metric_rows('CODEX holdout', label, info, ground_truth, prediction))
            if number == 1:
                codex_gallery.setdefault(record_id, {'samples': {}})['samples'][label] = {
                    'image': image_tensor.cpu(), 'ground_truth': ground_truth.cpu(),
                    'prediction': prediction, 'target_pixel_size_um': spec['requested_pixel_size_um'],
                }
            print(f'  [{number}/{len(codex_records)}] {record_id} complete', flush=True)
    finally:
        del bundle
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

CODEX_PER_RECORD_DF = pd.DataFrame(codex_rows)
CODEX_SUMMARY_DF, CODEX_THRESHOLD_DF = aggregate_metrics(CODEX_PER_RECORD_DF)
CODEX_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CODEX_PER_RECORD_DF.to_csv(CODEX_RESULTS_ROOT / 'per_record.csv', index=False)
CODEX_SUMMARY_DF.to_csv(CODEX_RESULTS_ROOT / 'summary.csv', index=False)
CODEX_THRESHOLD_DF.to_csv(CODEX_RESULTS_ROOT / 'thresholds.csv', index=False)
codex_manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'source': 'CPDMI_2023/CODEX', 'loader': 'load_CPDMI_CODEX',
    'n_records': len(codex_records),
    'nuclear_records': sum('nucleus_masks' in item for item in codex_records),
    'native_pixel_size_um': 0.377,
    'inference_tile_size_px': {spec['label']: spec['inference_tile_size_px'] for spec in MODEL_SPECS},
    'target_pixel_sizes_um': sorted({float(spec['requested_pixel_size_um']) for spec in MODEL_SPECS}),
    'device': CODEX_DEVICE,
    'models': MODEL_SPECS,
    'postprocessing_reference': POSTPROCESSING,
    'postprocessing_by_model': MODEL_POSTPROCESSING,
    'records': [str(item['filename']) for item in codex_records],
}
(CODEX_RESULTS_ROOT / 'provenance.json').write_text(json.dumps(codex_manifest, indent=2) + '\n')
print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes.')
display(CODEX_SUMMARY_DF[['evaluation_set', 'model', 'target', 'n_records', 'n_true', 'n_pred', 'f1_iou50', 'macro_f1_iou50']])

# Plot the independent CODEX curves beside the existing CPDMI-validation curves.
curve_tables = [CODEX_THRESHOLD_DF]
if 'THRESHOLD_DF' in globals() and not THRESHOLD_DF.empty:
    curve_tables.insert(0, THRESHOLD_DF[THRESHOLD_DF['evaluation_set'] == 'CPDMI validation'])
else:
    prior_thresholds_path = RESULTS_ROOT / 'thresholds.csv'
    if prior_thresholds_path.is_file():
        prior_thresholds = pd.read_csv(prior_thresholds_path)
        prior_thresholds = prior_thresholds[prior_thresholds['evaluation_set'] == 'CPDMI validation']
        curve_tables.insert(0, prior_thresholds)
curve_table = pd.concat(curve_tables, ignore_index=True)
figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for axis, target in zip(axes, TARGET_NAMES):
    data = curve_table[curve_table['target'] == target]
    for (evaluation_set, model), curve in data.groupby(['evaluation_set', 'model']):
        axis.plot(curve['threshold'], curve['f1'], marker='o', label=f'{evaluation_set}: {model}')
    axis.set_title(target.title())
    axis.set_xlabel('IoU threshold')
    axis.set_ylabel('pooled/object-count F1')
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.25)
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
figure.tight_layout()
plt.show()

display(codex_thresholds := CODEX_THRESHOLD_DF[CODEX_THRESHOLD_DF['threshold'] == 0.5][[
    'model', 'target', 'n_records', 'n_true', 'n_pred', 'f1', 'macro_f1'
]])
print('Interpretation: the nuclear CODEX comparison has only two annotated crops. If the public advantage persists there, it argues against CPDMI-validation memorization; if it disappears, leakage becomes a hypothesis, but platform/domain shift must still be ruled out.')


## Tiling sensitivity diagnostics for CPDMI-050 versus public

This section is separate from the main model-ranking and CODEX cells. It tests whether the nuclear F1 difference is caused or amplified by input tiling and label stitching. The default run uses the seven CPDMI validation records that contain nuclear annotations, which keeps the diagnostic manageable and focused on the current nuclear discrepancy. Set `TILING_USE_ALL_VALIDATION_RECORDS = True` to include all 30 validation records for cell metrics.

The diagnostic compares one padded whole-field input with several tiling configurations. It also tests the public model's embedded postprocessing defaults and an optional trained-predictor path that resolves cell/nucleus boundaries per tile, so the shared tiler is not mistaken for identical downstream processing.

Set `RUN_TILING_DIAGNOSTIC = True` and run the next cell on a CUDA kernel. Results are written under `model_comparisons/.../tiling_sensitivity/` without overwriting the main comparison outputs.

In [ ]:
# This is intentionally gated so running the original notebook does not launch a second evaluation.
RUN_TILING_DIAGNOSTIC = True
TILING_USE_ALL_VALIDATION_RECORDS = False
TILING_INCLUDE_PUBLIC_NATIVE = True
TILING_RESULTS_ROOT = RESULTS_ROOT / 'tiling_sensitivity'

TILING_CONFIGS = [
    {'name': 'whole_field', 'tile_size_px': None, 'overlap': None, 'max_cell_size': None},
    {'name': 'tile256_current', 'tile_size_px': 256, 'overlap': 80, 'max_cell_size': 20},
    {'name': 'tile256_overlap80_maxcell0', 'tile_size_px': 256, 'overlap': 80, 'max_cell_size': 0},
    {'name': 'tile256_overlap64_maxcell0', 'tile_size_px': 256, 'overlap': 64, 'max_cell_size': 0},
    {'name': 'tile384_overlap80_maxcell20', 'tile_size_px': 384, 'overlap': 80, 'max_cell_size': 20},
    {'name': 'tile384_overlap64_maxcell0', 'tile_size_px': 384, 'overlap': 64, 'max_cell_size': 0},
]

class _ResolvedTrainedTilePredictor(nn.Module):
    """Apply the same explicit per-tile biological resolution as the public predictor."""
    def __init__(self, base_predictor):
        super().__init__()
        self.base_predictor = base_predictor

    def forward(self, image_batch, resolve_cell_and_nucleus=True, **kwargs):
        labels = self.base_predictor(
            image_batch, resolve_cell_and_nucleus=False, **kwargs
        )
        if resolve_cell_and_nucleus and labels.ndim == 4 and labels.shape[1] == 2:
            labels = loss_utils.resolve_cell_and_nucleus_boundaries(labels.float())
        return labels

def _tiling_public_native_kwargs(network):
    return {
        'mask_threshold': float(network.default_mask_threshold),
        'peak_distance': int(network.default_peak_distance),
        'seed_threshold': float(network.default_seed_threshold),
        'overlap_threshold': float(network.default_overlap_threshold),
        'mean_threshold': float(network.default_mean_threshold),
        'window_size': int(network.default_window_size),
        'min_size': int(network.default_min_size),
        'cleanup_fragments': bool(network.default_cleanup_fragments),
        'max_seeds': 2000,
        'resolve_cell_and_nucleus': bool(network.default_resolve_cell_and_nucleus),
    }

def _predict_tiling_diagnostic(bundle, image_tensor, tiling_config, postprocessing, predictor):
    model_input, original_shape = _pad_for_model(image_tensor)
    kwargs = dict(postprocessing)
    with torch.inference_mode():
        if tiling_config['tile_size_px'] is None:
            prediction = predictor(
                model_input[None].to(bundle['device']), **kwargs
            )[0]
        else:
            prediction = _sliding_window_inference(
                model_input, predictor,
                window_size=(tiling_config['tile_size_px'], tiling_config['tile_size_px']),
                overlap=tiling_config['overlap'],
                max_cell_size=tiling_config['max_cell_size'],
                sw_device=bundle['device'], device='cpu', batch_size=1,
                output_channels=2, show_progress=False,
                instanseg_kwargs=kwargs,
            ).squeeze(0)
    prediction = prediction.detach().to('cpu').to(torch.int32)
    return _crop_prediction(prediction, original_shape)

TILING_TARGET_NAMES = ('nuclei', 'cells')

def _tiling_evaluation_arrays(gt, prediction, target_index):
    gt_eval = gt[target_index].clone().to(torch.int32)
    prediction_eval = prediction[target_index].clone().to(torch.int32)
    valid = gt_eval >= 0
    if not bool(valid.all()):
        gt_eval = gt_eval.clone()
        prediction_eval = prediction_eval.clone()
        gt_eval[~valid] = 0
        prediction_eval[~valid] = 0
    if not bool((gt_eval > 0).any()):
        return None
    return gt_eval, prediction_eval

def _tiling_metric_rows(model_label, predictor_mode, postprocessing_mode, tiling_name, info, gt, prediction):
    rows = []
    for target_index, target_name in enumerate(TILING_TARGET_NAMES):
        arrays = _tiling_evaluation_arrays(gt, prediction, target_index)
        if arrays is None:
            continue
        gt_eval, prediction_eval = arrays
        for threshold, stat in zip(THRESHOLDS, matching_torch(gt_eval, prediction_eval, THRESHOLDS)):
            rows.append({
                'evaluation_set': 'CPDMI validation tiling sensitivity',
                'model': model_label, 'predictor_mode': predictor_mode,
                'postprocessing_mode': postprocessing_mode, 'tiling': tiling_name,
                'record_id': info['record_id'], 'filename': info['filename'],
                'target': target_name, 'threshold': float(threshold),
                'tp': int(stat.tp), 'fp': int(stat.fp), 'fn': int(stat.fn),
                'precision': float(stat.precision), 'recall': float(stat.recall),
                'f1': float(stat.f1), 'n_true': int(stat.n_true),
                'n_pred': int(stat.n_pred),
            })
    return rows

def _aggregate_tiling_metrics(per_record):
    if per_record.empty:
        return pd.DataFrame(), pd.DataFrame()
    group_columns = ['model', 'predictor_mode', 'postprocessing_mode', 'tiling', 'target', 'threshold']
    rows = []
    for keys, group in per_record.groupby(group_columns, sort=True):
        model, predictor_mode, postprocessing_mode, tiling, target, threshold = keys
        tp, fp, fn = (int(group[name].sum()) for name in ('tp', 'fp', 'fn'))
        rows.append({
            'model': model, 'predictor_mode': predictor_mode,
            'postprocessing_mode': postprocessing_mode, 'tiling': tiling,
            'target': target, 'threshold': threshold, 'tp': tp, 'fp': fp, 'fn': fn,
            'precision': tp / max(tp + fp, 1e-10),
            'recall': tp / max(tp + fn, 1e-10),
            'f1': 2 * tp / max(2 * tp + fp + fn, 1e-10),
            'macro_f1': float(group['f1'].mean()),
            'n_true': int(group['n_true'].sum()), 'n_pred': int(group['n_pred'].sum()),
            'n_records': int(group['record_id'].nunique()),
        })
    threshold_table = pd.DataFrame(rows)
    summary = threshold_table[threshold_table['threshold'] == 0.5].copy()
    return summary, threshold_table

if not RUN_TILING_DIAGNOSTIC:
    print('Tiling diagnostic is gated. Set RUN_TILING_DIAGNOSTIC=True and rerun this cell on CUDA.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('The tiling diagnostic is intended for a CUDA notebook kernel.')

    if TILING_USE_ALL_VALIDATION_RECORDS:
        tiling_records = list(RECORD_GROUPS['CPDMI validation'])
    else:
        tiling_records = [
            (info, item) for info, item in RECORD_GROUPS['CPDMI validation']
            if info['nucleus_annotation']
        ]
    if not tiling_records:
        raise RuntimeError('No CPDMI validation records selected for tiling diagnostic.')

    tiling_model_specs = [MODEL_SPECS[0], MODEL_SPECS[-1]]
    prepared_tiling_records = [
        (info, item, *prepare_record(item, 0.5))
        for info, item in tiling_records
    ]
    print('Records:', len(prepared_tiling_records))
    print('Nuclear-annotated records:', sum(info['nucleus_annotation'] for info, _, _, _ in prepared_tiling_records))

    tiling_rows = []
    started = time.perf_counter()
    for spec in tiling_model_specs:
        print(f"[{spec['label']}] loading", flush=True)
        bundle = load_model_bundle(spec)
        try:
            predictor_modes = ['current']
            if spec['kind'] == 'trained':
                predictor_modes.append('per_tile_resolved')
            postprocessing_modes = ['shared']
            if spec['kind'] == 'public' and TILING_INCLUDE_PUBLIC_NATIVE:
                postprocessing_modes.append('public_native')
            for predictor_mode in predictor_modes:
                predictor = bundle['predictor']
                if predictor_mode == 'per_tile_resolved':
                    predictor = _ResolvedTrainedTilePredictor(predictor)
                for postprocessing_mode in postprocessing_modes:
                    if postprocessing_mode == 'public_native':
                        postprocessing = _tiling_public_native_kwargs(bundle['network'])
                    else:
                        postprocessing = MODEL_POSTPROCESSING[spec['label']]
                    for tiling_config in TILING_CONFIGS:
                        print(
                            f"  [{predictor_mode}/{postprocessing_mode}/{tiling_config['name']}]",
                            flush=True,
                        )
                        for number, (info, item, image_tensor, ground_truth) in enumerate(
                            prepared_tiling_records, start=1
                        ):
                            prediction = _predict_tiling_diagnostic(
                                bundle, image_tensor, tiling_config, postprocessing, predictor
                            )
                            tiling_rows.extend(
                                _tiling_metric_rows(
                                    spec['label'], predictor_mode, postprocessing_mode,
                                    tiling_config['name'], info, ground_truth, prediction
                                )
                            )
                        print(f"    {number}/{len(prepared_tiling_records)}", flush=True)
        finally:
            del bundle
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    TILING_PER_RECORD_DF = pd.DataFrame(tiling_rows)
    TILING_SUMMARY_DF, TILING_THRESHOLD_DF = _aggregate_tiling_metrics(TILING_PER_RECORD_DF)
    print(f"Completed in {(time.perf_counter() - started) / 60:.1f} minutes.")
    display(TILING_SUMMARY_DF[[
        'model', 'predictor_mode', 'postprocessing_mode', 'tiling', 'target',
        'n_records', 'n_true', 'n_pred', 'precision', 'recall', 'f1', 'macro_f1'
    ]].sort_values(['target', 'model', 'predictor_mode', 'postprocessing_mode', 'tiling']))

    TILING_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    TILING_PER_RECORD_DF.to_csv(TILING_RESULTS_ROOT / 'per_record.csv', index=False)
    TILING_SUMMARY_DF.to_csv(TILING_RESULTS_ROOT / 'summary.csv', index=False)
    TILING_THRESHOLD_DF.to_csv(TILING_RESULTS_ROOT / 'thresholds.csv', index=False)
    tiling_manifest = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'source': 'CPDMI validation', 'n_records': len(prepared_tiling_records),
        'models': tiling_model_specs, 'tiling_configs': TILING_CONFIGS,
        'postprocessing_shared': POSTPROCESSING,
        'include_public_native': TILING_INCLUDE_PUBLIC_NATIVE,
        'use_all_validation_records': TILING_USE_ALL_VALIDATION_RECORDS,
        'records': [info for info, _, _, _ in prepared_tiling_records],
    }
    (TILING_RESULTS_ROOT / 'provenance.json').write_text(
        json.dumps(tiling_manifest, indent=2, default=str) + '\n'
    )

    plot_data = TILING_SUMMARY_DF
    figure, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    tiling_order = [config['name'] for config in TILING_CONFIGS]
    for axis, target in zip(axes, TILING_TARGET_NAMES):
        target_data = plot_data[plot_data['target'] == target]
        for keys, curve in target_data.groupby(
            ['model', 'predictor_mode', 'postprocessing_mode'], sort=False
        ):
            model, predictor_mode, postprocessing_mode = keys
            curve = curve.set_index('tiling').reindex(tiling_order).reset_index()
            axis.plot(
                curve['tiling'], curve['f1'], marker='o',
                label=f'{model} | {predictor_mode} | {postprocessing_mode}',
            )
        axis.set_title(target.title())
        axis.set_xlabel('Inference configuration')
        axis.set_ylabel('Pooled F1 at IoU 0.5')
        axis.tick_params(axis='x', rotation=45)
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.25)
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    figure.tight_layout()
    plt.show()
    print('Saved tiling diagnostic artifacts to:', TILING_RESULTS_ROOT)


## Training-aligned independent nuclear-head assessment

The main comparison uses `resolve_cell_and_nucleus=True` and reports the paired biological output. This cell adds the complementary training-process metric with `resolve_cell_and_nucleus=False`. It produces the full dual-head output with independent nuclear and cell stitching, and evaluates the nuclear head from `prediction[0]`; it does not rerun a separate resolved-nucleus score because, when resolving, the nucleus is the primary object and the paired result is already reported above.

Set `RUN_INDEPENDENT_NUCLEUS_EVALUATION = True` and run this cell on CUDA. The default uses the same CPDMI validation and bounded saved-Test records as the main comparison.

In [ ]:
RUN_INDEPENDENT_NUCLEUS_EVALUATION = True
INDEPENDENT_NUCLEUS_USE_SAVED_TEST = True
INDEPENDENT_NUCLEUS_RESULTS_ROOT = RESULTS_ROOT / 'independent_nuclei'
INDEPENDENT_NUCLEUS_POSTPROCESSING = {**POSTPROCESSING, 'resolve_cell_and_nucleus': False}
INDEPENDENT_NUCLEUS_POSTPROCESSING_BY_MODEL = {
    spec['label']: {**MODEL_POSTPROCESSING[spec['label']], 'resolve_cell_and_nucleus': False}
    for spec in MODEL_SPECS
}

def _independent_nucleus_arrays(gt, prediction):
    gt_eval = gt[0].clone().to(torch.int32)
    # output_channels=2 retains both independently stitched heads; score nuclei from channel 0.
    prediction_eval = (prediction[0] if prediction.ndim == 3 else prediction).clone().to(torch.int32)
    valid = gt_eval >= 0
    if not bool(valid.all()):
        gt_eval = gt_eval.clone()
        prediction_eval = prediction_eval.clone()
        gt_eval[~valid] = 0
        prediction_eval[~valid] = 0
    if not bool((gt_eval > 0).any()):
        return None
    return gt_eval, prediction_eval

def _predict_independent_nuclei(bundle, image_tensor, spec, postprocessing):
    model_input, original_shape = _pad_for_model(image_tensor)
    with torch.inference_mode():
        prediction = _sliding_window_inference(
            model_input, bundle['predictor'],
            window_size=(spec['inference_tile_size_px'], spec['inference_tile_size_px']),
            overlap=80, max_cell_size=20,
            sw_device=bundle['device'], device='cpu', batch_size=1,
            output_channels=2, show_progress=False,
            instanseg_kwargs=dict(postprocessing),
        ).squeeze(0)
    return _crop_prediction(prediction.detach().to('cpu').to(torch.int32), original_shape)

def _aggregate_independent_nuclei(per_record):
    if per_record.empty:
        return pd.DataFrame(), pd.DataFrame()
    group_columns = ['evaluation_set', 'model', 'threshold']
    rows = []
    for keys, group in per_record.groupby(group_columns, sort=True):
        evaluation_set, model, threshold = keys
        tp, fp, fn = (int(group[name].sum()) for name in ('tp', 'fp', 'fn'))
        rows.append({
            'evaluation_set': evaluation_set, 'model': model, 'target': 'nuclei',
            'threshold': threshold, 'tp': tp, 'fp': fp, 'fn': fn,
            'precision_iou50': tp / max(tp + fp, 1e-10),
            'recall_iou50': tp / max(tp + fn, 1e-10),
            'f1_iou50': 2 * tp / max(2 * tp + fp + fn, 1e-10),
            'macro_f1_iou50': float(group['f1'].mean()),
            'n_true': int(group['n_true'].sum()), 'n_pred': int(group['n_pred'].sum()),
            'n_records': int(group['record_id'].nunique()),
        })
    threshold_table = pd.DataFrame(rows)
    return threshold_table[threshold_table['threshold'] == 0.5].copy(), threshold_table

if not RUN_INDEPENDENT_NUCLEUS_EVALUATION:
    print('Independent nuclear-head assessment is gated. Set RUN_INDEPENDENT_NUCLEUS_EVALUATION=True and rerun this cell on CUDA.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('The independent nuclear-head assessment is intended for a CUDA notebook kernel.')
    independent_split_names = ['CPDMI validation']
    if INDEPENDENT_NUCLEUS_USE_SAVED_TEST:
        independent_split_names.append('Saved Test split')
    independent_records = [
        (split_name, info, item)
        for split_name in independent_split_names
        for info, item in RECORD_GROUPS.get(split_name, [])
    ]
    if not independent_records:
        raise RuntimeError('No records selected for independent nuclear-head assessment.')
    print('Records by split:', pd.Series([row[0] for row in independent_records]).value_counts().to_dict())

    independent_rows = []
    started = time.perf_counter()
    for spec in MODEL_SPECS:
        print(f"[{spec['label']}] loading", flush=True)
        bundle = load_model_bundle(spec)
        try:
            for number, (split_name, info, item) in enumerate(independent_records, start=1):
                if not info['nucleus_annotation']:
                    continue
                image_tensor, ground_truth = prepare_record(
                    item, spec['requested_pixel_size_um']
                )
                prediction = _predict_independent_nuclei(
                    bundle, image_tensor, spec,
                    INDEPENDENT_NUCLEUS_POSTPROCESSING_BY_MODEL[spec['label']],
                )
                arrays = _independent_nucleus_arrays(ground_truth, prediction)
                if arrays is None:
                    continue
                gt_eval, prediction_eval = arrays
                for threshold, stat in zip(
                    THRESHOLDS, matching_torch(gt_eval, prediction_eval, THRESHOLDS)
                ):
                    independent_rows.append({
                        'evaluation_set': split_name, 'model': spec['label'],
                        'record_id': info['record_id'], 'filename': info['filename'],
                        'threshold': float(threshold), 'tp': int(stat.tp),
                        'fp': int(stat.fp), 'fn': int(stat.fn),
                        'precision': float(stat.precision), 'recall': float(stat.recall),
                        'f1': float(stat.f1), 'n_true': int(stat.n_true),
                        'n_pred': int(stat.n_pred),
                    })
                if number == 1 or number % 25 == 0 or number == len(independent_records):
                    print(f'  {number}/{len(independent_records)}', flush=True)
        finally:
            del bundle
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    INDEPENDENT_NUCLEUS_PER_RECORD_DF = pd.DataFrame(independent_rows)
    INDEPENDENT_NUCLEUS_SUMMARY_DF, INDEPENDENT_NUCLEUS_THRESHOLD_DF = _aggregate_independent_nuclei(
        INDEPENDENT_NUCLEUS_PER_RECORD_DF
    )
    print(f"Completed in {(time.perf_counter() - started) / 60:.1f} minutes.")
    display(INDEPENDENT_NUCLEUS_SUMMARY_DF[[
        'evaluation_set', 'model', 'target', 'n_records', 'n_true', 'n_pred',
        'precision_iou50', 'recall_iou50', 'f1_iou50', 'macro_f1_iou50'
    ]].sort_values(['evaluation_set', 'model']))

    INDEPENDENT_NUCLEUS_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    INDEPENDENT_NUCLEUS_PER_RECORD_DF.to_csv(
        INDEPENDENT_NUCLEUS_RESULTS_ROOT / 'per_record.csv', index=False
    )
    INDEPENDENT_NUCLEUS_SUMMARY_DF.to_csv(
        INDEPENDENT_NUCLEUS_RESULTS_ROOT / 'summary.csv', index=False
    )
    INDEPENDENT_NUCLEUS_THRESHOLD_DF.to_csv(
        INDEPENDENT_NUCLEUS_RESULTS_ROOT / 'thresholds.csv', index=False
    )

    resolved_summary = SUMMARY_DF.copy() if 'SUMMARY_DF' in globals() else pd.DataFrame()
    if resolved_summary.empty and (RESULTS_ROOT / 'summary.csv').is_file():
        resolved_summary = pd.read_csv(RESULTS_ROOT / 'summary.csv')
    resolved_summary = resolved_summary[
        (resolved_summary.get('target', pd.Series(dtype=str)) == 'nuclei')
        & (resolved_summary.get('threshold', pd.Series(dtype=float)) == 0.5)
    ].copy()
    resolved_summary['protocol'] = 'resolve=True paired'
    resolved_summary = resolved_summary.rename(columns={
        'precision_iou50': 'precision', 'recall_iou50': 'recall',
        'f1_iou50': 'f1', 'macro_f1_iou50': 'macro_f1',
    })
    independent_summary = INDEPENDENT_NUCLEUS_SUMMARY_DF.copy()
    independent_summary['protocol'] = 'resolve=False independent nuclei'
    independent_summary = independent_summary.rename(columns={
        'precision_iou50': 'precision', 'recall_iou50': 'recall',
        'f1_iou50': 'f1', 'macro_f1_iou50': 'macro_f1',
    })
    comparison_columns = [
        'evaluation_set', 'model', 'protocol', 'n_records', 'n_true', 'n_pred',
        'precision', 'recall', 'f1', 'macro_f1',
    ]
    display(pd.concat([
        resolved_summary[comparison_columns], independent_summary[comparison_columns]
    ], ignore_index=True).sort_values(['evaluation_set', 'model', 'protocol']))
    independent_manifest = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'protocol': 'resolve=False independent nuclear head',
        'models': MODEL_SPECS,
        'postprocessing_reference': INDEPENDENT_NUCLEUS_POSTPROCESSING,
        'postprocessing_by_model': INDEPENDENT_NUCLEUS_POSTPROCESSING_BY_MODEL,
        'inference_tile_size_px': {spec['label']: spec['inference_tile_size_px'] for spec in MODEL_SPECS},
        'records': [info for _, info, _ in independent_records if info['nucleus_annotation']],
    }
    (INDEPENDENT_NUCLEUS_RESULTS_ROOT / 'provenance.json').write_text(
        json.dumps(independent_manifest, indent=2, default=str) + '\n'
    )
    print('Saved independent-head artifacts to:', INDEPENDENT_NUCLEUS_RESULTS_ROOT)
